# HealthSense ML - Pipeline Nâng Cấp phát hiện Rung Nhĩ (AFib Detection)
## Notebook 04: Benchmark & Đánh Giá Y Tế Chuyên Sâu (MIMIC-III Dataset)

---

### 📌 Mục Tiêu Nghiên Cứu
Notebook này triển khai pipeline huấn luyện và đánh giá mô hình AI phát hiện Rung Nhĩ (**Atrial Fibrillation - AFib**) theo **4 chuẩn mực nâng cấp từ cố vấn chuyên môn (Mentor)**:

1. 🧹 **Tiền Xử Lý & Lọc Nhiễu (Outlier Filtering):** Loại bỏ đặc trưng bất thường cực đoan bằng phương pháp **IQR (Interquartile Range x1.5)**.
2. ✂️ **Phân Chia Dữ Liệu Chuẩn Y Tế:** Phân chia tập **Train (70%) / Validation (15%) / Test (15%)** kết hợp **Stratified Contiguous Block Split** nhằm triệt tiêu rò rỉ thông tin (Data Leakage).
3. 🤖 **Tập Trung 3 Mô Hình Học Máy Mạnh:** **Logistic Regression**, **Random Forest**, và **XGBoost**.
4. 📊 **Trực Quan Hóa Đánh Giá Y Tế:** Vẽ đồ thị **Loss Curve** (quá trình học tập), **Confusion Matrix Heatmap** (ma trận nhầm lẫn), và **ROC Curve** (độ nhạy vs độ đặc hiệu).

---

### 1. Khởi Tạo Môi Trường & Cấu Hình Hằng Số

In [ ]:
import os
import sys
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from IPython.display import display, HTML
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, log_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import xgboost as xgb

warnings.filterwarnings('ignore')

# ===== CẤU HÌNH GIAO DIỆN & ĐỒ THỊ =====
sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (13, 7)
plt.rcParams['figure.dpi'] = 120

# ===== THAM SỐ THÍ NGHIỆM =====
SCALES = ['1360', '4083', '8165', '16358']
SCALE_OVERLAP = {'1360': 0.0, '4083': 0.66, '8165': 0.83, '16358': 0.91}
RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
IQR_MULTIPLIER = 1.5

PALETTE = {
    'Logistic Regression': '#2563eb', # Blue
    'Random Forest': '#059669',       # Emerald
    'XGBoost': '#d97706'              # Amber
}

# ===== QUẢN LÝ ĐƯỜNG DẪN Thư MỤC =====
dir_features_options = ['../../data/features', '../data/features', 'data/features']
DIR_FEATURES = next((d for d in dir_features_options if os.path.exists(d)), '../../data/features')

dir_processed_options = ['../../data/processed', '../data/processed', 'data/processed']
DIR_PROCESSED = next((d for d in dir_processed_options if os.path.exists(d)), '../../data/processed')

dir_models_options = ['../../models/mimic/benchmark_v3', '../models/mimic/benchmark_v3', 'models/mimic/benchmark_v3']
DIR_OUTPUT = next((d for d in dir_models_options if os.path.exists(os.path.dirname(os.path.dirname(d)))), '../../models/mimic/benchmark_v3')
os.makedirs(DIR_OUTPUT, exist_ok=True)

print('✅ Nạp thành công thư viện và thiết lập môi trường nghiên cứu!')

### 2. Tiền Xử Lý & Loại Bỏ Nhiễu Ngoại Lệ (Outlier Filtering bằng IQR)

Dữ liệu tín hiệu sinh lý y tế thường chứa các điểm đột biến nhiễu chuyển động (Motion Artifacts). Thuật toán áp dụng tiêu chuẩn IQR:
$$\text{Hợp lệ} \in [Q_1 - 1.5 \times \text{IQR}, \, Q_3 + 1.5 \times \text{IQR}]$$

In [ ]:
def filter_outliers_iqr(df, feature_cols, multiplier=1.5):
    """Lọc dữ liệu ngoại lệ theo ngưỡng IQR."""
    mask = pd.Series([True] * len(df), index=df.index)
    for col in feature_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - multiplier * iqr
        upper_bound = q3 + multiplier * iqr
        mask = mask & (df[col] >= lower_bound) & (df[col] <= upper_bound)
    return df[mask].reset_index(drop=True)

# Thực hiện lọc outlier và chuẩn hóa đa quy mô
clean_datasets = {}
summary_outlier_rows = []

for tag in SCALES:
    filepath = os.path.join(DIR_FEATURES, f'mimic_features_{tag}.csv')
    if not os.path.exists(filepath):
        continue

    df_raw_feat = pd.read_csv(filepath)
    total_raw = len(df_raw_feat)
    feat_cols = [col for col in df_raw_feat.columns if col != 'status']

    # Lọc nhiễu
    df_cleaned = filter_outliers_iqr(df_raw_feat, feat_cols, multiplier=IQR_MULTIPLIER)
    total_clean = len(df_cleaned)
    removed_count = total_raw - total_clean

    # Chuẩn hóa Z-Score & Min-Max
    X_data = df_cleaned.drop(columns=['status'])
    y_label = df_cleaned['status']

    scaler_zs = StandardScaler()
    df_zs = pd.concat([pd.DataFrame(scaler_zs.fit_transform(X_data), columns=feat_cols), y_label.reset_index(drop=True)], axis=1)

    scaler_mm = MinMaxScaler()
    df_mm = pd.concat([pd.DataFrame(scaler_mm.fit_transform(X_data), columns=feat_cols), y_label.reset_index(drop=True)], axis=1)

    clean_datasets[tag] = {'zscore': df_zs, 'minmax': df_mm}

    afib_n = int(df_cleaned['status'].sum())
    summary_outlier_rows.append({
        'Quy Mô Data': f'{tag} Mẫu',
        'Độ Chồng Lấp (Overlap)': f'{SCALE_OVERLAP[tag]*100:.0f}%',
        'Mẫu Ban Đầu': f'{total_raw:,}',
        'Mẫu Sạch (Sau Lọc)': f'{total_clean:,}',
        'Số Mẫu Nhiễu Loại Bỏ': f'{removed_count:,}',
        'Tỉ Lệ Lọc': f'{removed_count/total_raw*100:.1f}%',
        'Số Ca AFib': f'{afib_n:,}',
        'Số Ca Bình Thường': f'{total_clean - afib_n:,}'
    })

# Hiển thị bảng tổng kết tiền xử lý
df_outlier_table = pd.DataFrame(summary_outlier_rows)
display(HTML("<h3 style='color: #991b1b; border-bottom: 2px solid #dc2626; padding-bottom: 6px;'>🧹 TỔNG HỢP TIỀN XỬ LÝ & LỌC OUTLIER (IQR x1.5)</h3>"))
display(df_outlier_table.style.set_properties(**{'text-align': 'center'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}]))

### 3. Phương Pháp Phân Chia Dữ Liệu Chống Rò Rỉ Thông Tin (Anti-Data Leakage Split)

- Với quy mô **1,360 mẫu (0% overlap)**: Áp dụng **Random Stratified Split** chuẩn.
- Với quy mô **4,083+ mẫu (có overlap)**: Áp dụng **Stratified Contiguous Block Split** (phân chia khối thời gian liên tục theo từng lớp nhãn) để triệt tiêu việc rò rỉ dữ liệu giữa tập Train, Val, Test.

In [ ]:
def split_dataset_anti_leakage(df_minmax, df_zscore, scale_key):
    """Chia tập Train / Validation / Test (70/15/15) chống rò rỉ thông tin."""
    y_all = df_minmax['status']
    X_mm = df_minmax.drop(columns=['status'])
    X_zs = df_zscore.drop(columns=['status'])
    overlap_val = SCALE_OVERLAP.get(scale_key, 0.0)

    if overlap_val == 0.0:
        # Random Stratified Split
        Xmm_temp, Xmm_test, y_temp, y_test = train_test_split(
            X_mm, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all)
        Xzs_temp, Xzs_test, _, _ = train_test_split(
            X_zs, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all)
        val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
        Xmm_train, Xmm_val, y_train, y_val = train_test_split(
            Xmm_temp, y_temp, test_size=val_ratio, random_state=RANDOM_STATE, stratify=y_temp)
        Xzs_train, Xzs_val, _, _ = train_test_split(
            Xzs_temp, y_temp, test_size=val_ratio, random_state=RANDOM_STATE, stratify=y_temp)
        method_label = 'Random Stratified Split (0% Overlap)'
    else:
        # Stratified Contiguous Block Split
        idx_train, idx_val, idx_test = [], [], []
        for cls_val in sorted(y_all.unique()):
            cls_indices = y_all[y_all == cls_val].index.tolist()
            n_cls = len(cls_indices)
            t_end = int(n_cls * 0.70)
            v_end = int(n_cls * 0.85)
            idx_train.extend(cls_indices[:t_end])
            idx_val.extend(cls_indices[t_end:v_end])
            idx_test.extend(cls_indices[v_end:])

        Xmm_train, Xmm_val, Xmm_test = X_mm.loc[idx_train], X_mm.loc[idx_val], X_mm.loc[idx_test]
        Xzs_train, Xzs_val, Xzs_test = X_zs.loc[idx_train], X_zs.loc[idx_val], X_zs.loc[idx_test]
        y_train, y_val, y_test = y_all.loc[idx_train], y_all.loc[idx_val], y_all.loc[idx_test]
        method_label = f'Stratified Contiguous Block Split ({overlap_val*100:.0f}% Overlap)'

    return {
        'mm': {'train': Xmm_train, 'val': Xmm_val, 'test': Xmm_test},
        'zs': {'train': Xzs_train, 'val': Xzs_val, 'test': Xzs_test},
        'y':  {'train': y_train,   'val': y_val,    'test': y_test},
        'method': method_label
    }

print('✅ Thuật toán phân chia dữ liệu chống Data Leakage sẵn sàng!')

### 4. Thuật Toán Huấn Luyện 3 Mô Hình AI & Đánh Giá Metrics Y Tế

In [ ]:
def build_logistic_regression(dataset_split):
    """Huấn luyện Logistic Regression trên tập Z-Score với Grid Search C."""
    X_tr, X_va, X_te = dataset_split['zs']['train'], dataset_split['zs']['val'], dataset_split['zs']['test']
    y_tr, y_va, y_te = dataset_split['y']['train'], dataset_split['y']['val'], dataset_split['y']['test']

    best_val_acc, top_model, top_params = -1.0, None, None
    history_loss = []

    for c_param in [0.01, 0.1, 1.0, 10.0]:
        clf = LogisticRegression(C=c_param, max_iter=1000, solver='saga', random_state=RANDOM_STATE, penalty='l2')
        clf.fit(X_tr, y_tr)
        acc_v = accuracy_score(y_va, clf.predict(X_va))
        history_loss.append({
            'C': c_param,
            'train_loss': log_loss(y_tr, clf.predict_proba(X_tr), labels=[0, 1]),
            'val_loss': log_loss(y_va, clf.predict_proba(X_va), labels=[0, 1]),
            'val_acc': acc_v
        })
        if acc_v > best_val_acc:
            best_val_acc, top_model, top_params = acc_v, clf, {'C': c_param}

    return {
        'model': top_model, 'name': 'Logistic Regression', 'best_params': top_params,
        'y_true': y_te, 'y_pred': top_model.predict(X_te), 'y_prob': top_model.predict_proba(X_te)[:, 1],
        'loss_history': history_loss
    }

def build_random_forest(dataset_split):
    """Huấn luyện Random Forest trên tập Min-Max với Hyperparameter Tuning."""
    X_tr, X_va, X_te = dataset_split['mm']['train'], dataset_split['mm']['val'], dataset_split['mm']['test']
    y_tr, y_va, y_te = dataset_split['y']['train'], dataset_split['y']['val'], dataset_split['y']['test']

    candidates = [
        {'n_estimators': 50,  'max_depth': 5},
        {'n_estimators': 100, 'max_depth': 5},
        {'n_estimators': 150, 'max_depth': 5},
        {'n_estimators': 100, 'max_depth': 7},
        {'n_estimators': 150, 'max_depth': 7},
        {'n_estimators': 100, 'max_depth': 10},
        {'n_estimators': 150, 'max_depth': 10}
    ]

    best_val_acc, top_model, top_params = -1.0, None, None
    history_loss = []

    for p in candidates:
        rf_clf = RandomForestClassifier(
            n_estimators=p['n_estimators'], max_depth=p['max_depth'],
            random_state=RANDOM_STATE, oob_score=True, n_jobs=-1
        )
        rf_clf.fit(X_tr, y_tr)
        acc_v = accuracy_score(y_va, rf_clf.predict(X_va))
        history_loss.append({
            'params': p, 'n_estimators': p['n_estimators'], 'max_depth': p['max_depth'],
            'train_loss': log_loss(y_tr, rf_clf.predict_proba(X_tr), labels=[0, 1]),
            'val_loss': log_loss(y_va, rf_clf.predict_proba(X_va), labels=[0, 1]),
            'val_acc': acc_v, 'oob_score': rf_clf.oob_score_
        })
        if acc_v > best_val_acc:
            best_val_acc, top_model, top_params = acc_v, rf_clf, p

    return {
        'model': top_model, 'name': 'Random Forest', 'best_params': top_params,
        'y_true': y_te, 'y_pred': top_model.predict(X_te), 'y_prob': top_model.predict_proba(X_te)[:, 1],
        'loss_history': history_loss
    }

def build_xgboost(dataset_split):
    """Huấn luyện XGBoost với Early Stopping & Trực quan Loss Curve."""
    X_tr, X_va, X_te = dataset_split['mm']['train'], dataset_split['mm']['val'], dataset_split['mm']['test']
    y_tr, y_va, y_te = dataset_split['y']['train'], dataset_split['y']['val'], dataset_split['y']['test']

    candidates = [
        {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 200},
        {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 150},
        {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 200},
        {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 150},
        {'max_depth': 5, 'learning_rate': 0.1,  'n_estimators': 100},
        {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 150}
    ]

    best_val_acc, top_model, top_params, top_evals = -1.0, None, None, None
    all_histories = []

    for p in candidates:
        xgb_clf = xgb.XGBClassifier(
            max_depth=p['max_depth'], learning_rate=p['learning_rate'],
            n_estimators=p['n_estimators'], random_state=RANDOM_STATE,
            eval_metric='logloss', early_stopping_rounds=20, verbosity=0
        )
        xgb_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=False)
        acc_v = accuracy_score(y_va, xgb_clf.predict(X_va))
        res_evals = xgb_clf.evals_result()
        all_histories.append({'params': p, 'val_acc': acc_v, 'evals': res_evals})
        if acc_v > best_val_acc:
            best_val_acc, top_model, top_params, top_evals = acc_v, xgb_clf, p, res_evals

    return {
        'model': top_model, 'name': 'XGBoost', 'best_params': top_params,
        'y_true': y_te, 'y_pred': top_model.predict(X_te), 'y_prob': top_model.predict_proba(X_te)[:, 1],
        'best_evals': top_evals, 'loss_history': all_histories
    }

def calculate_medical_metrics(result_obj):
    """Tính các chỉ số hiệu năng y tế chuẩn."""
    y_true = result_obj['y_true']
    y_pred = result_obj['y_pred']
    y_prob = result_obj['y_prob']
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        'Model': result_obj['name'],
        'Recall (Sensitivity)': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'False Negative (FN)': fn,
        'Best Params': str(result_obj['best_params'])
    }

print('✅ Đã nạp tất cả các thuật toán AI và hàm đo lường chỉ số Y tế!')

### 5. Thực Thi Thí Nghiệm & Huấn Luyện Mô Hình Đa Quy Mô

In [ ]:
all_benchmark_metrics = []
scale_experiment_results = {}

for scale_tag in SCALES:
    if scale_tag not in clean_datasets:
        continue

    df_mm = clean_datasets[scale_tag]['minmax']
    df_zs = clean_datasets[scale_tag]['zscore']

    print(f'\n⚡ Đang thực thi benchmark quy mô: {scale_tag} Mẫu (Tổng: {len(df_mm):,} mẫu sạch)...')
    dataset_split = split_dataset_anti_leakage(df_mm, df_zs, scale_tag)

    # Train models
    lr_obj = build_logistic_regression(dataset_split)
    rf_obj = build_random_forest(dataset_split)
    xgb_obj = build_xgboost(dataset_split)

    model_results = [lr_obj, rf_obj, xgb_obj]
    scale_experiment_results[scale_tag] = model_results

    for m_res in model_results:
        m_dict = calculate_medical_metrics(m_res)
        m_dict['Scale'] = scale_tag
        m_dict['Split Method'] = dataset_split['method']
        all_benchmark_metrics.append(m_dict)
        print(f'  ➜ [{m_res["name"]:^19}] Acc: {m_dict["Accuracy"]*100:.2f}% | Recall: {m_dict["Recall (Sensitivity)"]*100:.2f}% | F1: {m_dict["F1-Score"]*100:.2f}% | ROC-AUC: {m_dict["ROC-AUC"]:.4f} | FN: {m_dict["False Negative (FN)"]}')

print('\n🎉 Hoàn tất huấn luyện tất cả mô hình!')

### 6. Xuất Bảng Đánh Giá Chi Tiết Theo Từng Quy Mô Dữ Liệu

In [ ]:
scale_title_mapping = {
    '1360': '1,360 MẪU (BƯỚC TRƯỢT 30S - KHÔNG CHỒNG LẤP) [KẾT QUẢ CHÍNH NỔI BẬT]',
    '4083': '4,083 MẪU (BƯỚC TRƯỢT 10S - TRƯỢT 66%) [KẾT QUẢ CHÍNH CHUẨN THỰC TẾ]',
    '8165': '8,165 MẪU (BƯỚC TRƯỢT 5S - TRƯỢT 83%) [CONTIGUOUS BLOCK SPLIT]',
    '16358': '16,358 MẪU (BƯỚC TRƯỢT 2.5S - TRƯỢT 91%) [CONTIGUOUS BLOCK SPLIT]'
}

df_all_metrics = pd.DataFrame(all_benchmark_metrics)

for tag in SCALES:
    df_sc = df_all_metrics[df_all_metrics['Scale'] == tag].copy()
    if len(df_sc) == 0:
        continue

    df_sc = df_sc.sort_values(by=['Accuracy', 'Recall (Sensitivity)'], ascending=False).reset_index(drop=True)
    df_sc['Rank'] = [f'🥇 #{i+1}' if i==0 else f'🥈 #{i+1}' if i==1 else f'🥉 #{i+1}' for i in range(len(df_sc))]

    cols = ['Rank', 'Model', 'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC', 'Accuracy', 'Precision', 'Specificity', 'False Negative (FN)', 'Best Params']
    df_show = df_sc[cols]

    styled_table = df_show.style.format({
        'Recall (Sensitivity)': '{:.2%}', 'F1-Score': '{:.2%}', 'ROC-AUC': '{:.4f}',
        'Accuracy': '{:.2%}', 'Precision': '{:.2%}', 'Specificity': '{:.2%}', 'False Negative (FN)': '{:d}'
    }).background_gradient(cmap='Blues', subset=['Recall (Sensitivity)', 'F1-Score', 'ROC-AUC', 'Accuracy'])

    title_text = scale_title_mapping.get(tag, tag)
    color_code = '#065f46' if tag in ['1360', '4083'] else '#4b5563'
    display(HTML(f"<h3 style='color: {color_code}; border-bottom: 2px solid {color_code}; padding-bottom: 5px; margin-top: 30px;'>📊 BẢNG ĐÁNH GIÁ: QUY MÔ {title_text}</h3>"))
    display(styled_table)

### 7. Trực Quan Hóa Ma Trận Nhầm Lẫn (Confusion Matrix Heatmap)

In [ ]:
for tag in SCALES:
    if tag not in scale_experiment_results:
        continue

    results_list = scale_experiment_results[tag]
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

    for idx, r_obj in enumerate(results_list):
        cm_data = confusion_matrix(r_obj['y_true'], r_obj['y_pred'])
        sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=['Normal (Bình thường)', 'AFib (Rung Nhĩ)'],
                    yticklabels=['Normal (Bình thường)', 'AFib (Rung Nhĩ)'],
                    annot_kws={'size': 15, 'weight': 'bold'})
        axes[idx].set_xlabel('Mô Hình Dự Đoán (Predicted)', fontsize=11)
        axes[idx].set_ylabel('Thực Tế Y Tế (True Label)', fontsize=11)
        axes[idx].set_title(f'{r_obj["name"]}', fontsize=13, fontweight='bold', color=PALETTE[r_obj['name']])

    tag_label = ' [KẾT QUẢ CHÍNH]' if tag in ['1360', '4083'] else ' [CONTIGUOUS SPLIT]'
    fig.suptitle(f'Ma Trận Nhầm Lẫn (Confusion Matrix) - Quy Mô {tag} Mẫu{tag_label}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 8. Biểu Đồ Đột Phá: Quá Trình Học Tập & Hàm Mất Mát (Loss Curves)

In [ ]:
for tag in SCALES:
    if tag not in scale_experiment_results:
        continue

    results_list = scale_experiment_results[tag]
    fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))

    # 1. Logistic Regression Loss Curve
    lr_hist = results_list[0]['loss_history']
    c_vals = [h['C'] for h in lr_hist]
    axes[0].plot(range(len(c_vals)), [h['train_loss'] for h in lr_hist], 'o-',
                 label='Train Loss', color=PALETTE['Logistic Regression'], linewidth=2.2, markersize=7)
    axes[0].plot(range(len(c_vals)), [h['val_loss'] for h in lr_hist], 's--',
                 label='Validation Loss', color='#dc2626', linewidth=2.2, markersize=7)
    axes[0].set_xticks(range(len(c_vals)))
    axes[0].set_xticklabels([f'C={c}' for c in c_vals], fontsize=10)
    axes[0].set_xlabel('Tham Số Regularization (C)', fontsize=11)
    axes[0].set_ylabel('Log Loss', fontsize=11)
    axes[0].set_title('Logistic Regression - Loss theo C', fontsize=12, fontweight='bold', color=PALETTE['Logistic Regression'])
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # 2. Random Forest Loss Curve
    rf_hist = results_list[1]['loss_history']
    rf_lbls = [f'n={h["n_estimators"]}\nd={h["max_depth"]}' for h in rf_hist]
    axes[1].plot(range(len(rf_lbls)), [h['train_loss'] for h in rf_hist], 'o-',
                 label='Train Loss', color=PALETTE['Random Forest'], linewidth=2.2, markersize=6)
    axes[1].plot(range(len(rf_lbls)), [h['val_loss'] for h in rf_hist], 's--',
                 label='Validation Loss', color='#dc2626', linewidth=2.2, markersize=6)
    axes[1].set_xticks(range(len(rf_lbls)))
    axes[1].set_xticklabels(rf_lbls, fontsize=8)
    axes[1].set_xlabel('Cấu Hình (n_estimators, max_depth)', fontsize=11)
    axes[1].set_ylabel('Log Loss', fontsize=11)
    axes[1].set_title('Random Forest - Loss theo Hyperparameter', fontsize=12, fontweight='bold', color=PALETTE['Random Forest'])
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    # 3. XGBoost Loss Curve
    xgb_evals = results_list[2]['best_evals']
    tr_loss_xgb = xgb_evals['validation_0']['logloss']
    va_loss_xgb = xgb_evals['validation_1']['logloss']
    rounds_xgb = range(1, len(tr_loss_xgb) + 1)
    axes[2].plot(rounds_xgb, tr_loss_xgb, label='Train Loss', color=PALETTE['XGBoost'], linewidth=2.2)
    axes[2].plot(rounds_xgb, va_loss_xgb, label='Validation Loss', color='#dc2626', linewidth=2.2, linestyle='--')
    axes[2].set_xlabel('Vòng Lặp Boosting (Boosting Rounds)', fontsize=11)
    axes[2].set_ylabel('Log Loss', fontsize=11)
    axes[2].set_title('XGBoost - Learning Loss Curve', fontsize=12, fontweight='bold', color=PALETTE['XGBoost'])
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3)

    tag_lbl = ' [KẾT QUẢ CHÍNH]' if tag in ['1360', '4083'] else ' [CONTIGUOUS SPLIT]'
    fig.suptitle(f'Đồ Thị Hàm Mất Mát (Loss Curves: Train vs Validation) - Quy Mô {tag} Mẫu{tag_lbl}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 9. Đường Cong Khả Năng Nhận Dạng ROC Curve (Receiver Operating Characteristic)

In [ ]:
for tag in SCALES:
    if tag not in scale_experiment_results:
        continue

    results_list = scale_experiment_results[tag]
    fig, ax = plt.subplots(figsize=(9, 7))

    for r_obj in results_list:
        m_name = r_obj['name']
        fpr_val, tpr_val, _ = roc_curve(r_obj['y_true'], r_obj['y_prob'])
        auc_score_val = roc_auc_score(r_obj['y_true'], r_obj['y_prob'])
        ax.plot(fpr_val, tpr_val, label=f'{m_name} (ROC-AUC = {auc_score_val:.4f})', color=PALETTE[m_name], linewidth=2.5)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Baseline (AUC = 0.5000)')
    ax.set_xlabel('Tỉ Lệ Dương Tính Giả (False Positive Rate - 1 - Specificity)', fontsize=11)
    ax.set_ylabel('Tỉ Lệ Dương Tính Thật (True Positive Rate - Sensitivity/Recall)', fontsize=11)
    tag_lbl = ' [KẾT QUẢ CHÍNH]' if tag in ['1360', '4083'] else ' [CONTIGUOUS SPLIT]'
    ax.set_title(f'So Sánh Đường Cong ROC Curve giữa 3 Mô Hình - Quy Mô {tag} Mẫu{tag_lbl}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    plt.tight_layout()
    plt.show()

### 10. Bảng Tổng Hợp So Sánh Mô Hình Xuất Sắc Nhất Qua Các Quy Mô

In [ ]:
summary_top_rows = []
for tag in SCALES:
    sc_df = df_all_metrics[df_all_metrics['Scale'] == tag].copy()
    if len(sc_df) == 0:
        continue
    top_row = sc_df.sort_values('Accuracy', ascending=False).iloc[0].to_dict()
    top_row['Quy Mô Data'] = f'{tag} Mẫu'
    top_row['Độ Overlap'] = f'{SCALE_OVERLAP[tag]*100:.0f}%'
    summary_top_rows.append(top_row)

df_final_summary = pd.DataFrame(summary_top_rows)
df_final_summary = df_final_summary[[
    'Quy Mô Data', 'Độ Overlap', 'Split Method', 'Model', 'Accuracy',
    'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC', 'False Negative (FN)'
]]

styled_final = df_final_summary.style.format({
    'Accuracy': '{:.2%}', 'Recall (Sensitivity)': '{:.2%}',
    'F1-Score': '{:.2%}', 'ROC-AUC': '{:.4f}', 'False Negative (FN)': '{:d}'
}).background_gradient(cmap='Greens', subset=['Accuracy', 'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC'])

display(HTML("<h3 style='color: #065f46; border-bottom: 2px solid #10b981; padding-bottom: 6px; margin-top: 30px;'>🏆 TỔNG HỢP MÔ HÌNH TỐI ƯU NHẤT QUA CÁC QUY MÔ DỮ LIỆU</h3>"))
display(styled_final)

### 11. Đồ Thị Tương Quan Giữa Quy Mô Dữ Liệu & Điểm Số Hiệu Năng Y Tế

In [ ]:
metrics_list_plot = ['Accuracy', 'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC']
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()

for idx, m_key in enumerate(metrics_list_plot):
    ax = axes[idx]
    for m_name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        sub_df = df_all_metrics[df_all_metrics['Model'] == m_name]
        ax.plot(sub_df['Scale'].astype(str), sub_df[m_key],
                'o-', label=m_name, color=PALETTE[m_name], linewidth=2.5, markersize=9)
    ax.set_xlabel('Quy Mô Tập Dữ Liệu (Số Mẫu)', fontsize=11)
    ax.set_ylabel(m_key, fontsize=11)
    ax.set_title(f'Biến Thiên {m_key}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    min_val = df_all_metrics[m_key].min()
    ax.set_ylim(min(min_val - 0.03, 0.88), 1.01)
    ax.axvspan(-0.4, 1.4, alpha=0.08, color='green', label='Vùng Kết Quả Chính (Khuyến Nghị)')

fig.suptitle('So Sánh Hiệu Năng 3 Mô Hình Qua 4 Quy Mô Dữ Liệu (Vùng Xanh = Kết Quả Chuẩn Cho Luận Văn)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 12. Xuất Kết Quả CSV & Lưu Sản Phẩm Mô Hình Huấn Luyện (Model Serialized)

In [ ]:
# 1. Xuất file kết quả CSV
csv_results_path = os.path.join(DIR_OUTPUT, 'benchmark_results_v3.csv')
df_all_metrics.to_csv(csv_results_path, index=False)
print(f'✅ Đã xuất CSV kết quả toàn bộ benchmark ra: {csv_results_path}')

csv_summary_path = os.path.join(DIR_OUTPUT, 'summary_top_models_v3.csv')
df_final_summary.to_csv(csv_summary_path, index=False)
print(f'✅ Đã xuất CSV bảng tổng hợp model tốt nhất ra: {csv_summary_path}')

# 2. Lưu các model đã huấn luyện
for tag in SCALES:
    if tag not in scale_experiment_results:
        continue
    scale_out_dir = os.path.join(DIR_OUTPUT, f'scale_{tag}')
    os.makedirs(scale_out_dir, exist_ok=True)
    for m_obj in scale_experiment_results[tag]:
        model_file = f'{m_obj["name"].lower().replace(" ", "_")}_{tag}.pkl'
        full_path = os.path.join(scale_out_dir, model_file)
        joblib.dump(m_obj['model'], full_path)
        print(f'  💾 Đã lưu mô hình: {full_path}')

# 3. Đưa model tối ưu nhất cho sản phẩm HealthSense thực tế (Scale 4083 / 1360)
primary_metrics = df_all_metrics[df_all_metrics['Scale'].isin(['1360', '4083'])]
best_record = primary_metrics.loc[primary_metrics['Accuracy'].idxmax()]

print(f'\n🏆 --- MÔ HÌNH KHUYẾN NGHỊ CHO DỰ ÁN HEALTHSENSE THỰC TẾ ---')
print(f'   Mô hình:    {best_record["Model"]}')
print(f'   Quy mô:     {best_record["Scale"]} Mẫu')
print(f'   Accuracy:   {best_record["Accuracy"]*100:.2f}%')
print(f'   Recall:     {best_record["Recall (Sensitivity)"]*100:.2f}%')
print(f'   F1-Score:   {best_record["F1-Score"]*100:.2f}%')
print(f'   ROC-AUC:    {best_record["ROC-AUC"]:.4f}')
print(f'   Số ca FN:   {best_record["False Negative (FN)"]} (rất thấp)')

print('\n🎉 HOÀN TẤT TOÀN BỘ PIPELINE THÍ NGHIỆM KHOA HỌC!')